In [1]:
import pandas as pd


# ベースの特徴量生成クラス
class BaseFeature:
    def __init__(self, df: pd.DataFrame, col_name):
        self.df = df.copy()

        # タイムスタンプをdatetime型に変換
        self.df['time_stamp'] = pd.to_datetime(self.df['time_stamp'])

        # 曜日を追加
        self.df['weekday'] = self.df['time_stamp'].dt.day_name()

        # 減衰率カラムを追加
        self._attenuation_rate()

        # 関連度のカラム名
        self.rcol = col_name
    
    # 指定したカラム(ユーザIDか商品ID)毎に行動種別の集計(但し関連度を基準)
    def relevance_count(self, tarcol, type):
        # 指定された行動種別で絞る
        filtered_data = self.df[self.df[self.rcol] == type]

        # 作成する特徴量のカラム名を作成
        feature_name = f"{tarcol}_count_{type}"

        # 集計結果を返す
        return filtered_data.groupby([tarcol, self.rcol]).size().reset_index(name=feature_name)
    
    # 指定したカラム(ユーザIDか商品ID)毎に曜日単位で行動種別の集計(但し関連度を基準)
    def weekday_count(self, tarcol, type, wday):
        # 指定された行動種別でフィルタリング
        filtered_data = self.df[self.df[self.rcol] == type]

        # 指定された曜日でフィルタリング
        filtered_data = self.df[self.df['weekday'] == wday]

        # 作成する特徴量のカラム名を作成
        feature_name = f"{tarcol}_{wday}_count_{type}"
        
        # 集計結果を返す
        return filtered_data.groupby([tarcol, 'weekday']).size().reset_index(name=feature_name)
    
    # 減衰率の設定
    def _attenuation_rate(self):
        # 減衰率の設定
        r_values = [0.95, 0.9, 0.8]

        # 基準日（学習データの最終日）
        end_date = pd.to_datetime("2017-04-30")

        # time_stampをdatetime型に変換し、日付を抽出
        self.df['date'] = pd.to_datetime(self.df['time_stamp']).dt.date
        self.df['date'] = pd.to_datetime(self.df['date'])

        # 各減衰率ごとに重みを計算
        for r in r_values:
            self.df[f'weight_r_{r}'] = self.df['date'].apply(lambda x: r ** (end_date - x).days)


# ユーザ基準の特徴量生成クラス
class UserFeature(BaseFeature):
    def relevance_count(self):
        for i in range(4):
            count_data = super().relevance_count('user_id', i)

            # 結合と欠損値処理
            self.df = pd.merge(self.df, count_data, on=['user_id', 'event_type'], how='left').fillna(0)

# データ読み込み
sample_data = pd.DataFrame({
    'user_id': ['0000000_A', '0000000_A', '0000000_A', '0000000_A', '0000000_A', '0000000_A', '0000000_A', '0000000_A', '0000000_A', '0000000_A', '0000000_A', '0000000_A', '0000000_A', '0000000_A', '0000000_A', '0000642_A', '0000642_A', '0000642_A', '0000642_A'],
    'product_id': ['00009250_a', '00009250_a', '00014068_a', '00001254_a', '00003316_a', '00003009_a', '00004433_a', '00004433_a', '00008525_a', '00009753_a', '00009753_a', '00013303_a', '00003524_a', '00012725_a', '00008263_a', '00009250_a', '00003009_a', '00010627_a', '00009816_a'],
    'event_type': [0, 1, 2, 3, 2, 1, 3, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    'ad': [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    'time_stamp': ['2017-04-08 12:09:04', '2017-04-27 12:55:57', '2017-04-08 11:57:53', '2017-04-08 12:04:26', '2017-04-08 12:05:31', '2017-04-14 13:55:07', '2017-04-27 13:30:18', '2017-04-27 13:03:18', '2017-04-27 12:59:09', '2017-04-27 13:01:27', '2017-04-27 13:23:55', '2017-04-27 13:29:54', '2017-04-27 13:26:46', '2017-04-27 13:25:28', '2017-04-27 12:56:42', '2017-04-05 04:31:04', '2017-04-05 05:03:28', '2017-04-09 13:56:18', '2017-04-06 03:25:29']
})

In [3]:
# 特徴量計算
user_event_feature = UserFeature(sample_data, 'event_type')
print("ユーザごとのevent_type合計:")
user_event_feature.relevance_count()
print(user_event_feature.df)


ユーザごとのevent_type合計:
      user_id  product_id  event_type  ad          time_stamp    weekday  \
0   0000000_A  00009250_a           0  -1 2017-04-08 12:09:04   Saturday   
1   0000000_A  00009250_a           1  -1 2017-04-27 12:55:57   Thursday   
2   0000000_A  00014068_a           2  -1 2017-04-08 11:57:53   Saturday   
3   0000000_A  00001254_a           3  -1 2017-04-08 12:04:26   Saturday   
4   0000000_A  00003316_a           2  -1 2017-04-08 12:05:31   Saturday   
5   0000000_A  00003009_a           1  -1 2017-04-14 13:55:07     Friday   
6   0000000_A  00004433_a           3  -1 2017-04-27 13:30:18   Thursday   
7   0000000_A  00004433_a           0  -1 2017-04-27 13:03:18   Thursday   
8   0000000_A  00008525_a           1  -1 2017-04-27 12:59:09   Thursday   
9   0000000_A  00009753_a           1  -1 2017-04-27 13:01:27   Thursday   
10  0000000_A  00009753_a           1  -1 2017-04-27 13:23:55   Thursday   
11  0000000_A  00013303_a           1  -1 2017-04-27 13:29:54   Thur

In [ ]:
user_event_by_weekday_feature = UserFeature(sample_data, 'event_type')
print("ユーザごとの曜日単位event_type合計:")
print(user_event_by_weekday_feature.weekday_count())

In [ ]:
# 特徴量計算
user_event_feature = ItemFeature(sample_data, 'event_type')
print("ユーザごとのevent_type合計:")
print(user_event_feature.relevance_count())


In [ ]:
user_event_by_weekday_feature = ItemFeature(sample_data, 'event_type')
print("ユーザごとの曜日単位event_type合計:")
print(user_event_by_weekday_feature.weekday_count())